# What this simulation is doing

This notebook runs a **2D compressible Euler** model (`CompressibleFluid2D`) in a periodic box.

- **State variables**: `rho` (density), `u` (x-velocity), `v` (y-velocity), `p` (pressure).
- **Numerics**: conservative finite-volume style update with local Lax–Friedrichs (Rusanov) flux + adaptive CFL time stepping.
- **Boundary conditions**: **periodic in both x and y** (flow leaving one side re-enters from the opposite side).

## Initial-condition scenarios

The simulator now supports `scenario=`:

- `"shear_layers"`: two horizontal shear layers with multimode compressible perturbations.
- `"vortex_sheet"`: a single displaced shear sheet that rolls up into vortical structures.
- `"blast_wave"`: central over-pressure pulse producing outgoing compressible waves.

`amp` controls perturbation strength and `gamma` sets gas compressibility response.

So this notebook visualizes nonlinear coupling between advection, compression/expansion, and pressure-wave dynamics.

## Governing equations (2D Compressible Euler)

This simulator evolves conservative variables
$$
\mathbf{U}=[\rho,\ \rho u,\ \rho v,\ E]^\top
$$
with the 2D Euler system
$$
\frac{\partial \mathbf{U}}{\partial t} + \frac{\partial \mathbf{F}(\mathbf{U})}{\partial x} + \frac{\partial \mathbf{G}(\mathbf{U})}{\partial y} = 0.
$$

Fluxes are
$$
\mathbf{F}=\begin{bmatrix}
\rho u\\
\rho u^2+p\\
\rho uv\\
(E+p)u
\end{bmatrix},\qquad
\mathbf{G}=\begin{bmatrix}
\rho v\\
\rho uv\\
\rho v^2+p\\
(E+p)v
\end{bmatrix},
$$
with equation of state
$$
p=(\gamma-1)\left(E-\tfrac12\rho(u^2+v^2)\right).
$$

### Variable definitions

- $x,y$: spatial coordinates in the 2D domain.
- $t$: time.
- $\rho(x,y,t)$: mass density.
- $u(x,y,t), v(x,y,t)$: velocity components in $x$ and $y$.
- $p(x,y,t)$: thermodynamic pressure.
- $E(x,y,t)$: total energy density per unit volume,
  $$E = \frac{p}{\gamma-1} + \tfrac12\rho(u^2+v^2).$$
- $\gamma$: adiabatic index (ratio of specific heats), sampled as a simulator parameter.
- $\mathbf{U}$: conservative state vector $[\rho,\rho u,\rho v,E]^\top$.
- $\mathbf{F},\mathbf{G}$: physical flux vectors in the $x$ and $y$ directions.

Numerically, the notebook uses a finite-volume update with either LLF (Rusanov) or HLL interface fluxes on a periodic grid.

In [ ]:
from autosim.simulations import CompressibleFluid2D as Sim

sim = Sim(
    return_timeseries=True,
    log_level="warning",
    n=64,
    T=1.2,
    dt_save=0.01,
    cfl=0.32,
    scenario="vortex_sheet",
    parameters_range={
        "gamma": (1.4, 1.4),
        "amp": (0.14, 0.22),
    },
)

data = sim.forward_samples_spatiotemporal(n_samples=2, random_seed=42)

In [ ]:
# (batch, time, x, y, channels)
data["data"].shape

In [ ]:
from autosim.utils import plot_spatiotemporal_video

anim = plot_spatiotemporal_video(
    data["data"],
    batch_idx=1,
    channel_names=["rho", "u", "v", "p"],
)

In [ ]:
from IPython.display import HTML

HTML(anim.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt

# Compare final density field across all scenarios with identical parameters
scenarios = ["shear_layers", "vortex_sheet", "blast_wave"]
results = {}

for scenario_name in scenarios:
    sim_cmp = Sim(
        return_timeseries=False,
        log_level="warning",
        n=64,
        T=1.2,
        dt_save=0.01,
        cfl=0.32,
        scenario=scenario_name,
        parameters_range={
            "gamma": (1.4, 1.4),
            "amp": (0.18, 0.18),
        },
    )
    out = sim_cmp.forward_samples_spatiotemporal(n_samples=1, random_seed=42)
    # out["data"] shape: (1, 1, n, n, 4) because return_timeseries=False
    rho_final = out["data"][0, 0, :, :, 0]
    results[scenario_name] = rho_final

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, scenario_name in zip(axes, scenarios, strict=False):
    im = ax.imshow(results[scenario_name].cpu(), cmap="viridis", origin="lower")
    ax.set_title(scenario_name)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Final density ($\\rho$) comparison across scenarios", y=1.02)
plt.show()